### University of Virginia
### DS 7200: Distributed Computing
### Lab: Supervised Learning
### Last Updated: August 20, 2023

Sabine Segaloff

---

#### Instructions

This project has two parts:
- Part I: Classification - build and apply a logistic regression model on the Wisconsin Breast Cancer dataset.
- Part II: Regression - build and apply a linear regression model on the California Housing dataset.

**Total Possible Points: 10**

---

#### Part I: Classification (5 POINTS)

Here are the specifications and grading breakdown:

- the target variable is `diagnosis`
- use `f1`, `f2` as predictors (1 PT)
- split data into 60% training set, 40% test set 
- standardize the predictors (1 PT)
- use seed=314 whenever a seed is needed
- fit a Logistic Regression model with an intercept (1 PT)
- compute and show the area under the ROC curve for the test set (2 PTS)

In [1]:
from pyspark.sql import SparkSession

DATA_FILEPATH = 'wisc_breast_cancer_w_fields.csv'

spark = SparkSession \
    .builder \
    .appName("Wisc BRCA") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/02 23:05:29 WARN Utils: Your hostname, Beans-Legion, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/02 23:05:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/sabine/projects/distributed_computing/.venv/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/02 23:05:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


#### Enter code and solution

##### Load data

In [2]:
# load the data into DF

# inferSchema=True has Spark infer types instead of defaulting all to string
# note that it also makes the read not fully lazy (to guess the types Spark has to scan the file right away - this data is small enough that the extra pass over it isn't an issue)
cancer_df = spark.read.csv(DATA_FILEPATH, header=True, inferSchema=True)

# check that f1 and f2 came in as double (numeric) and that diagnosis is string
cancer_df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- diagnosis: string (nullable = true)
 |-- f1: double (nullable = true)
 |-- f2: double (nullable = true)
 |-- f3: double (nullable = true)
 |-- f4: double (nullable = true)
 |-- f5: double (nullable = true)
 |-- f6: double (nullable = true)
 |-- f7: double (nullable = true)
 |-- f8: double (nullable = true)
 |-- f9: double (nullable = true)
 |-- f10: double (nullable = true)
 |-- f11: double (nullable = true)
 |-- f12: double (nullable = true)
 |-- f13: double (nullable = true)
 |-- f14: double (nullable = true)
 |-- f15: double (nullable = true)
 |-- f16: double (nullable = true)
 |-- f17: double (nullable = true)
 |-- f18: double (nullable = true)
 |-- f19: double (nullable = true)
 |-- f20: double (nullable = true)
 |-- f21: double (nullable = true)
 |-- f22: double (nullable = true)
 |-- f23: double (nullable = true)
 |-- f24: double (nullable = true)
 |-- f25: double (nullable = true)
 |-- f26: double (nullable = true)
 |-- f27: double (

Spark's LogisticRegression requires a numeric label (0.0/1.0); it won't accept a string column. diagnosis is stored as "M" (malignant) / "B" (benign), so I need to convert it.

I'm choosing malignant = 1 since it's the event we care about detecting, and the evaluator treats 1.0 as the positive class.

**Note:** this is a fixed rule, not something learned from the data, so it's safe to do before the train/test split. No leakage (unlike the scaler, which learns means/SDs and must be fit on train only).

In [3]:
# check what values diagnosis actually takes before converting
# (otherwise(0.0) below would silently label any typo/missing value as benign)
cancer_df.groupBy("diagnosis").count().show()  # .show() is an action so Spark actually runs here

+---------+-----+
|diagnosis|count|
+---------+-----+
|        B|  357|
|        M|  212|
+---------+-----+



In [4]:
from pyspark.sql import functions as F  # F is the normal alias for Spark's column functions

# create numeric label column: malignant (M) = 1.0, benign (B) = 0.0
cancer_df = cancer_df.withColumn(           # withColumn returns a NEW DataFrame (Spark DFs are immutable), so reassign
    "label",                                # "label" is the default labelCol LogisticRegression looks for
    F.when(F.col("diagnosis") == "M", 1.0)  # if M -> 1.0
     .otherwise(0.0)                        # everything else -> 0.0 (safe to do since groupBy above showed only M and B)
)
# lazy: nothing runs yet (this just adds a step to the plan)

# confirm the mapping across all rows
# each diagnosis should map to exactly one label value
cancer_df.groupBy("diagnosis", "label").count().show()

+---------+-----+-----+
|diagnosis|label|count|
+---------+-----+-----+
|        B|  0.0|  357|
|        M|  1.0|  212|
+---------+-----+-----+



Each diagnosis maps to exactly one label (M to 1.0 and B to 0.0), and the counts match the original 212/357 so my conversion is correct.

##### Assemble predictors

Spark models only accept a single vector column for the predictors, so `VectorAssembler` packs `f1` and `f2` into one `raw_features` column.

`VectorAssembler` is a transformer with no `.fit()`, so it could safely run before the split. I'm putting it in the pipeline anyway so that all preprocessing lives in one place.

In [5]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=["f1", "f2"], # inputCols takes a list of column names (note plural: inputCols)
                            outputCol="raw_features")

cancer_vec_df = assembler.transform(cancer_df) # preview only - the pipeline does the real transform
cancer_vec_df.select("f1", "f2", "raw_features", "label").show(3, truncate=False) # action (spark runs here)

+-----+-----+-------------+-----+
|f1   |f2   |raw_features |label|
+-----+-----+-------------+-----+
|17.99|10.38|[17.99,10.38]|1.0  |
|20.57|17.77|[20.57,17.77]|1.0  |
|19.69|21.25|[19.69,21.25]|1.0  |
+-----+-----+-------------+-----+
only showing top 3 rows


##### Train/test split

I'm splitting `cancer_df` (which has the label but no feature vector yet), because the pipeline will do the assembling and scaling after the split. That's what keeps the scaler from seeing test data. (**Leak prevention**)

`randomSplit([0.6, 0.4], seed=314)` doesn't take an exact 60% sample. Spark flips a weighted coin for each row independently, so the sizes will only be approximately 60/40. It also doesn't stratify, so I'll check the class balance in each piece.

In [6]:
# split into train (~60%) and test (~40%) with seed=314 per instructions

# randomSplit takes a list of weights (not necessarily just two; could do [.7, .15, .15] for a train/val/test in one call)
# and returns a LIST of DataFrames (one per weight), so we unpack into two names
train_df, test_df = cancer_df.randomSplit([0.6, 0.4], seed=314)

# cache train_df as it gets reused several times (count, groupBy, pipeline fit) 
# without cache, each action replays the whole plan from the CSV read onward
# see mllib_regression_with_caching.ipynb for more info
# NOTE: .cache() is itself lazy - it only marks the DF; the first action below (.count()) actually fills the cache
# at 569 rows this won't measurably speed anything up - practicing the habit from mllib_regression_with_caching
train_df.cache()
# cache test_df too
test_df.cache()   # also reused: count, groupBy, then transform + evaluate

# check actual sizes - .count() is an action, so Spark runs the plan here
n_train = train_df.count()
n_test = test_df.count()
print(f"train: {n_train}, test: {n_test}, total: {n_train + n_test}")  # total should be 569 (no rows lost or duplicated)
print(f"train fraction: {n_train / (n_train + n_test):.3f}")           # should be close to 0.6, not exact
print(f"test fraction: {n_test / (n_train + n_test):.3f}")           # should be close to 0.4, not exact

26/10/02 23:05:34 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


train: 354, test: 215, total: 569
train fraction: 0.622
test fraction: 0.378


In [7]:
# randomSplit doesn't stratify, so we should check the M/B balance in each piece
# (overall: 212/569 = ~37% malignant)
train_df.groupBy("label").count().show()
test_df.groupBy("label").count().show()

+-----+-----+
|label|count|
+-----+-----+
|  0.0|  216|
|  1.0|  138|
+-----+-----+

+-----+-----+
|label|count|
+-----+-----+
|  0.0|  141|
|  1.0|   74|
+-----+-----+



In [8]:
# proportion malignant in each split (mean of a 0/1 column = proportion of 1s)
train_df.agg(F.mean("label").alias("prop_malignant")).show()
test_df.agg(F.mean("label").alias("prop_malignant")).show()

+------------------+
|    prop_malignant|
+------------------+
|0.3898305084745763|
+------------------+

+-------------------+
|     prop_malignant|
+-------------------+
|0.34418604651162793|
+-------------------+



The split gave 354 train / 215 test (62.2% train), close to the 60/40 target. The proportion malignant is 39.0% in train and 34.4% in test, close to the overall 37%, so the lack of stratification isn't a problem here.

##### Pipeline stages
The pipeline will run three stages in order, each reading the column the previous one wrote:

| Stage | Type | Learns from data? | Reads → writes |
|---|---|---|---|
| `assembler` | transformer | no | `f1`, `f2` → `raw_features` |
| `scaler` | estimator | yes (mean, SD per feature) | `raw_features` → `features` |
| `lr` | estimator | yes (coefficients, intercept) | `features`, `label` → predictions |

Because `scaler` and `lr` learn from whatever data they're fit on, the whole pipeline gets fit on `train_df` only. The test set only ever gets `.transform()`ed with the training statistics.

In [9]:
from pyspark.ml.feature import StandardScaler
from pyspark.ml.classification import LogisticRegression

# scaler is an estimator: .fit() learns each feature's mean and SD (from train only, via the pipeline)
scaler = StandardScaler(inputCol="raw_features",  # reads the assembler's output
                        outputCol="features",     # writes the scaled vector under the name the model reads by default
                        withMean=True,            # subtract the mean (default is False); needed for true standardization
                        withStd=True)             # divide by SD (default True; put it here for clarity)
# withMean=True + withStd=True = (x - mean) / sd
# (withMean=False would only divide by SD which would be scaling, not standardizing)

# logistic regression is an estimator: .fit() learns the coefficients and intercept
lr = LogisticRegression(featuresCol="features",  # the SCALED vector (not raw_features)
                        labelCol="label",        # numeric 0/1 label I created (M = 1.0)
                        fitIntercept=True)       # True is already the default, but I put it here for clarity
# no regParam/elasticNetParam: we aren't doing regularization, so default (no penalty) applies

# note: defining these objects runs nothing; they're configured tools until the pipeline is fit

##### Fit pipeline

In [10]:
from pyspark.ml import Pipeline

pipeline = Pipeline(
    stages=[
        assembler,
        scaler,
        lr
    ]
)

pipeline_model = pipeline.fit(train_df) # .fit() is where spark is actually running; scaler learns mean/SD from train, then lr trains on the scaled train data

# pull the fitted logistic regression out of the pipeline (last stage)
lr_model = pipeline_model.stages[-1]   # stages are in the same order as the pipeline list

print("Intercept:", lr_model.intercept)       # nonzero intercept confirms fitIntercept=True worked
print("Coefficients:", lr_model.coefficients) # one per predictor: [f1, f2] (in scaled units)

Intercept: -0.4550729488456534
Coefficients: [3.5700446280343403,1.0818379381825267]


The coefficients are log-odds per 1 SD, since the features are standardized. 
- f1: 3.57 - A 1-SD increase in f1, holding f2 fixed, multiplies the odds of malignancy by $e^{3.57} \approx 35.5$
- f2: 1.08 - A 1-SD increase in f2, holding f1 fixed, multiplies the odds of malignancy by $e^{1.08} \approx 2.95$

Both are positive, so higher values of either feature are associated with malignancy, and f1 has the much larger effect. Because both are in SD units, we can compare them directly like this.

Intercept: -0.455. The nonzero intercept confirms the model was fit with an intercept. Since we centered, the intercept is the log-odds of malignancy for a sample at the *average* of f1 and f2. Converted: $1 / (1 + e^{0.455}) \approx 0.39$, a 39% predicted probability of malignancy at average f1 and f2.

##### Predict and evaluate

`pipeline_model.transform(test_df)` runs the test set through all three fitted stages using the *training* statistics (the scaler applies the train mean/SD; it doesn't recompute them on test). It adds three prediction columns:

| Column | What it holds |
|---|---|
| `rawPrediction` | continuous scores (log-odds) for [class 0, class 1] |
| `probability` | the same scores converted to probabilities for [class 0, class 1] |
| `prediction` | hard 0/1 label (threshold 0.5) |

AUC needs a **continuous score**, not hard labels. It measures how well the model *ranks* malignant cases above benign ones across every possible threshold. With only 0/1 predictions there's just one threshold, so the AUC comes out wrong. That's why we use the default `rawPrediction` column, not `prediction`.

In [11]:
# apply the fitted pipeline to the test set (transform only - nothing is re-fit)
test_pred_df = pipeline_model.transform(test_df)  # lazy: adds steps to the plan

# look at a few predictions next to the truth
test_pred_df.select("label", "probability", "prediction").show(5, truncate=False)  # action

+-----+------------------------------------------+----------+
|label|probability                               |prediction|
+-----+------------------------------------------+----------+
|0.0  |[0.9240174999811224,0.07598250001887763]  |0.0       |
|1.0  |[0.0014364608997719049,0.9985635391002281]|1.0       |
|0.0  |[0.8388312056356981,0.16116879436430187]  |0.0       |
|0.0  |[0.5748392470339003,0.4251607529660997]   |0.0       |
|0.0  |[0.790322784707739,0.20967721529226102]   |0.0       |
+-----+------------------------------------------+----------+
only showing top 5 rows


In [12]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(
    rawPredictionCol="rawPrediction",  # continuous scores (default) - NOT "prediction" (hard 0/1 labels give a wrong AUC)
    labelCol="label",                  # true 0/1 label (M = 1.0)
    metricName="areaUnderROC"          # default but added for clarity
)

test_auc = evaluator.evaluate(test_pred_df)  # action: Spark runs here, returns a single float
print(f"Test AUC: {test_auc:.4f}")

Test AUC: 0.9514


In [13]:
# training AUC comes for free from the fitted model's summary (computed on train_df during .fit())
train_auc = lr_model.summary.areaUnderROC
print(f"Train AUC: {train_auc:.4f}")

Train AUC: 0.9523


The test AUC is 0.9514. This means that the probability that the model scores a randomly chosen malignant case higher than a randomly chosen benign one is about 95%. Test AUC is 0.0009 lower than train AUC (0.9523). That small gap is evidence against overfitting and is also unsurprising. An unregularized model with only two predictors has little capacity to overfit. This is a single 60/40 split, so the test AUC has some sampling variability. Repeating the split several times and averaging could give us a more stable estimate.

#### Part II: Regression (5 POINTS)

In this project, you will work with the California Home Price dataset to train a regression model and predict median home prices. Here are the specifications and grading breakdown:

- Scale the response variable median_house_value, dividing by 100000 (1 PT)

- Split data into train set (80%), test set (20%) using seed=314 (1 PT)

- Add new predictor: `rooms_per_household`

- In the training set, select all of these features and standardize them: (1 PT)

feats = ["total_bedrooms", 
         "population", 
         "households", 
         "median_income", 
         "rooms_per_household"]

- Fit a linear regression model on the training set with these parameters:

  - maxIter=10
  - regParam=0.3
  - elasticNetParam=0.8  


- Compute the MSE on the test set (2 PTS)

In [14]:
import os
import pandas as pd

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [15]:
DATA_FILEPATH2 = 'cal_housing_data_preproc_w_header.txt'

#### Enter code and solution

##### Load data

In [16]:
# load the data into DF

housing_df = spark.read.csv(DATA_FILEPATH2, header=True, inferSchema=True) # same as part I: inferSchema forces an extra pass over the file

# check schema
housing_df.printSchema()

root
 |-- median_house_value: double (nullable = true)
 |-- median_income: double (nullable = true)
 |-- housing_median_age: double (nullable = true)
 |-- total_rooms: double (nullable = true)
 |-- total_bedrooms: double (nullable = true)
 |-- population: double (nullable = true)
 |-- households: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)



In [17]:
# check for nulls in every column
# (want to check because VectorAssembler errors on nulls by default)
housing_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)   # count rows where column c is null
    for c in housing_df.columns
]).show()

+------------------+-------------+------------------+-----------+--------------+----------+----------+--------+---------+
|median_house_value|median_income|housing_median_age|total_rooms|total_bedrooms|population|households|latitude|longitude|
+------------------+-------------+------------------+-----------+--------------+----------+----------+--------+---------+
|                 0|            0|                 0|          0|             0|         0|         0|       0|        0|
+------------------+-------------+------------------+-----------+--------------+----------+----------+--------+---------+



No missing values, so no imputation is needed.

In [18]:
# check for zero households before dividing (rooms_per_household = total_rooms / households)
print("rows with zero households:", housing_df.filter(F.col("households") == 0).count())

rows with zero households: 0


##### Transform target and engineer feature

I need to do two fixed-rule transformations. Both are safe to do before the split since neither learns anything from the data.

First I have to scale the target: `median_house_value / 100000`, so the target is in units of $100k. A value of 4.5 means $450,000. The MSE will be in those units squared, ($100k)².

Then I have to make a new predictor: `rooms_per_household = total_rooms / households`. This lab doesn't define this, so I will use the following as my definition: average rooms per household in the block group. (The check earlier showed no missing values and no zero households, so there's no divide-by-zero risk.)

In [19]:
# (F was already imported in Part I: from pyspark.sql import functions as F)

housing_df = (
    housing_df
    # scale target to units of $100k (overwrites the original column)
    .withColumn("median_house_value", F.col("median_house_value") / 100000)
    # engineered feature: average rooms per household
    .withColumn("rooms_per_household", F.col("total_rooms") / F.col("households")) # chained: each withColumn returns a new DF the next builds on
)
# lazy: nothing runs yet

# confirm the values appear correct
# first row should be: 452600 -> 4.526, and 880 / 126 rooms per household -> ~6.98
housing_df.select("median_house_value", "total_rooms", "households", "rooms_per_household").show(3)  # action

+------------------+-----------+----------+-------------------+
|median_house_value|total_rooms|households|rooms_per_household|
+------------------+-----------+----------+-------------------+
|             4.526|      880.0|     126.0|  6.984126984126984|
|             3.585|     7099.0|    1138.0|  6.238137082601054|
|             3.521|     1467.0|     177.0|  8.288135593220339|
+------------------+-----------+----------+-------------------+
only showing top 3 rows


##### Train/test split

80/20 split with seed=314, done after the fixed transformations and before anything that learns from the data (the scaler and the model). I use the same logic as Part I: the pipeline gets fit on `housing_train_df` only.

In [20]:
# split into train (~80%) and test (~20%), seed=314 per instructions
housing_train_df, housing_test_df = housing_df.randomSplit([0.8, 0.2], seed=314)

# cache both: reused for count, pipeline fit, and transform + evaluate
housing_train_df.cache()
housing_test_df.cache()

# check sizes - .count() is an action, and also fills the caches
n_train = housing_train_df.count()
n_test = housing_test_df.count()
print(f"train: {n_train}, test: {n_test}, total: {n_train + n_test}")  # total should be 20640
print(f"train fraction: {n_train / (n_train + n_test):.3f}")           # should be close to 0.8
print(f"test fraction: {n_test / (n_train + n_test):.3f}")           # should be close to 0.2

train: 16472, test: 4168, total: 20640
train fraction: 0.798
test fraction: 0.202


The split gave 16,472 train / 4,168 test (79.8% train, 20.2% test). This is very close to the 80/20 target, and much closer than Part I's 62.2% was to 60%. With 20,640 rows, the per-row coin flips average out, so the realized fraction lands near the target.

##### Pipeline stages

| Stage | Type | Learns from data? | Reads → writes |
|---|---|---|---|
| `housing_assembler` | transformer | no | 5 feature columns → `raw_features` |
| `housing_scaler` | estimator | yes (mean, SD per feature) | `raw_features` → `features` |
| `linreg` | estimator | yes (coefficients, intercept) | `features`, `median_house_value` → `prediction` |

**Regularization:** the model uses an elastic net penalty, a mix of lasso (L1) and ridge (L2).
- `regParam=0.3` is the overall penalty strength
- `elasticNetParam=0.8` is the L1/L2 mix: 0 = pure ridge, 1 = pure lasso, so 0.8 is mostly lasso
- Because it's mostly L1, some coefficients may be shrunk all the way to zero, which effectively drops those features.

In general, penalized regression requires features to be on comparable scales, or else the penalty hits them unevenly. Spark's `LinearRegression` handles this internally by default (see note below), so here my scaler is mainly satisfying the requirement and puts the reported coefficients into SD units.

In [21]:
# features specified (rooms_per_household is the one I engineered)
feats = ["total_bedrooms",
         "population",
         "households",
         "median_income",
         "rooms_per_household"]

# new stage objects with housing_ names (so Part I's assembler/scaler aren't overwritten)
housing_assembler = VectorAssembler(inputCols=feats,             # list of the 5 predictor columns
                                    outputCol="raw_features")

housing_scaler = StandardScaler(inputCol="raw_features",
                                outputCol="features",
                                withMean=True,                    # center (default False)
                                withStd=True)                     # scale to unit SD

# fit on housing_train_df only (via the pipeline)

from pyspark.ml.regression import LinearRegression

linreg = LinearRegression(featuresCol="features",
                          labelCol="median_house_value",  # target, already scaled to $100k units
                          maxIter=10,                     # max optimization iterations
                          regParam=0.3,                   # penalty strength, lambda
                          elasticNetParam=0.8)            # L1/L2 mix, alpha: 0.8 = mostly lasso
# fitIntercept=True is the default so we didn't have to specify

**Note:** Spark's `LinearRegression` has its own `standardization` parameter that defaults to `True`. It standardizes features internally before applying the penalty, then reports coefficients back in the input features' units. Since our inputs are already standardized, that internal step is effectively a no-op here. It's still worth knowing about, because it means that without our scaler, the penalty would still be applied fairly, but the reported coefficients would be in raw units.

##### Fit pipeline

In [22]:
housing_pipeline = Pipeline(
    stages=[
        housing_assembler,
        housing_scaler,
        linreg
    ]
)

housing_pipeline_model = housing_pipeline.fit(housing_train_df) # .fit() is where spark is actually running; scaler learns mean/SD from train, then linreg trains on the scaled train data

# pull the fitted linear regression out of the pipeline (last stage)
linreg_model = housing_pipeline_model.stages[-1]   # stages are in the same order as the pipeline list

print("Intercept:", linreg_model.intercept)       # nonzero intercept confirms fitIntercept=True worked
print("Coefficients:", linreg_model.coefficients) # one per predictor, in feats order (5 total; in SD units); sparse format: (size, [nonzero indices], [values])

Intercept: 2.070762241986439
Coefficients: (5,[3],[0.5228380318201065])


In [23]:
for name, coef in zip(feats, linreg_model.coefficients.toArray()):
    print(f"{name:>22}: {coef:.4f}")

        total_bedrooms: 0.0000
            population: 0.0000
            households: 0.0000
         median_income: 0.5228
   rooms_per_household: 0.0000


The lasso-heavy penalty zeroed every feature except median_income, whose coefficient (0.523) means about +$52k per 1-SD increase in income, shrunk toward zero. The intercept (2.07) is about $207k, the predicted value at average features.

##### Predict and evaluate

`housing_pipeline_model.transform(housing_test_df)` runs the test set through all three fitted stages using the training mean/SD, then adds a `prediction` column. MSE is the average squared difference between `prediction` and the true `median_house_value`, in units of ($100k)².

In [24]:
from pyspark.ml.evaluation import RegressionEvaluator

# apply fitted pipeline to the test set (transform only - nothing is re-fit)
housing_test_pred_df = housing_pipeline_model.transform(housing_test_df)  # lazy

# look at a few predictions next to the truth
housing_test_pred_df.select("median_house_value", "prediction").show(5)  # action

reg_evaluator = RegressionEvaluator(labelCol="median_house_value",  # true value (in $100k)
                                    predictionCol="prediction",     # model's prediction (default)
                                    metricName="mse")               # default is rmse!

test_mse = reg_evaluator.evaluate(housing_test_pred_df)  # action
print(f"Test MSE: {test_mse:.4f}")
print(f"Test RMSE: {test_mse ** 0.5:.4f}")  # sqrt puts error back in $100k units, easier to read

+------------------+------------------+
|median_house_value|        prediction|
+------------------+------------------+
|           0.14999|2.1593737893021085|
|             0.225|1.2181898007776142|
|              0.25|1.2362857681007073|
|             0.342| 1.294281406493985|
|              0.35|1.6585526753143576|
+------------------+------------------+
only showing top 5 rows
Test MSE: 0.7552
Test RMSE: 0.8690


The test MSE is 0.7552 (in ($100k)²). The RMSE of 0.869 is easier to read: predictions are typically off by about $87,000, which is large relative to the ~$207k average home value.

Note that `RegressionEvaluator`'s default metric is **RMSE, not MSE**, so `metricName="mse"` has to be set explicitly.

(The five rows shown above are the lowest-value homes, not a random sample. `randomSplit` sorts rows within partitions, so they come out roughly ordered by `median_house_value`. The model overpredicts these most, as expected for a shrunk, income-only fit.)

In [25]:
# training MSE comes for free from the fitted model's summary (computed on housing_train_df during .fit())
train_mse = linreg_model.summary.meanSquaredError
print(f"Train MSE: {train_mse:.4f}")
print(f"Train RMSE: {train_mse ** 0.5:.4f}")

# compare to test: 
# a small gap = no overfitting; 
# similar but high values on both = the model is underfitting (constrained by the penalty)
print(f"Test - Train MSE: {test_mse - train_mse:.4f}")

Train MSE: 0.7777
Train RMSE: 0.8819
Test - Train MSE: -0.0225


Train MSE (0.7777) and test MSE (0.7552) are nearly identical; test is even slightly lower, which is within the random variation of which rows landed in each split. So the model isn't overfitting. The high error on both sets instead points to underfitting. The strong, mostly-lasso penalty left an income-only model with a shrunk coefficient, too constrained to capture much of the variation in home values.